In [1]:
import requests
import pandas as pd

def download_and_convert(name1: str, name2: str):
    """
    Download two Lucene defect datasets from GitHub, convert to UTF-8,
    and return them as DataFrames.
    """

    def process_file(filename: str):
        base = (
            "https://raw.githubusercontent.com/"
            "awsm-research/line-level-defect-prediction/"
            "master/Dataset/File-level/"
        )

        url = base + filename

        resp = requests.get(url)
        if resp.status_code == 200:
            with open(filename, "wb") as f:
                f.write(resp.content)
        else:
            raise RuntimeError(f"Failed to download {filename}, status {resp.status_code}")

        df = pd.read_csv(filename, encoding="latin1")
        df.to_csv(filename, index=False, encoding="utf-8")
        return df

    return process_file(name1), process_file(name2)

In [2]:
# ------------------------------------------------------------
# DATASET LIST
# ------------------------------------------------------------
dataset_names = [
    ("lucene-2.9.0_ground-truth-files_dataset.csv","lucene-3.0.0_ground-truth-files_dataset.csv"),
    ("groovy-1_5_7_ground-truth-files_dataset.csv", "groovy-1_6_BETA_2_ground-truth-files_dataset.csv"),
    # ("hive-0.10.0_ground-truth-files_dataset.csv","hive-0.12.0_ground-truth-files_dataset.csv"),
    # ("derby-10.3.1.4_ground-truth-files_dataset.csv","derby-10.5.1.1_ground-truth-files_dataset.csv"),
    ("hbase-0.95.0_ground-truth-files_dataset.csv","hbase-0.95.2_ground-truth-files_dataset.csv"),
    ("camel-2.10.0_ground-truth-files_dataset.csv","camel-2.11.0_ground-truth-files_dataset.csv"),
    # ("activemq-5.3.0_ground-truth-files_dataset.csv","activemq-5.8.0_ground-truth-files_dataset.csv"),
    # ("wicket-1.3.0-incubating-beta-1_ground-truth-files_dataset.csv","wicket-1.5.3_ground-truth-files_dataset.csv"),
    ("jruby-1.4.0_ground-truth-files_dataset.csv", "jruby-1.5.0_ground-truth-files_dataset.csv"),
    ("derby-10.3.1.4_ground-truth-files_dataset.csv","derby-10.5.1.1_ground-truth-files_dataset.csv"),
    # ("hbase-0.95.0_ground-truth-files_dataset.csv","hbase-0.95.2_ground-truth-files_dataset.csv"),
]


# ------------------------------------------------------------
# STORAGE STRUCTURE
# ------------------------------------------------------------
dataset_pairs = []   # List of tuples: [(df_old, df_new, name1, name2), ...]

for name1, name2 in dataset_names:
    df_old, df_new = download_and_convert(name1, name2)
    dataset_pairs.append((df_old, df_new, name1, name2))

In [3]:
dataset_names_final = [name[0].split("-")[0] for name in dataset_names]

In [4]:
dataset_names_final

['lucene', 'groovy', 'hbase', 'camel', 'jruby', 'derby']

In [5]:
to_remove = ['activemq','wicket', 'hive', 'camel']
dataset_names_final = [dataset for dataset in dataset_names_final if dataset not in to_remove]

In [6]:
dataset_names_final

['lucene', 'groovy', 'hbase', 'jruby', 'derby']

In [7]:
for dfv1, dfv2,_,_ in dataset_pairs:
    dfv1 = dfv1.dropna(subset=['SRC']).reset_index(drop=True)
    dfv2 = dfv2.dropna(subset=['SRC']).reset_index(drop=True)

In [8]:
import pandas as pd
import re

def extract_version(filename):
    """Extract dataset name and version from the filename."""
    # Example: "lucene-2.9.0_ground-truth-files_dataset.csv"
    base = filename.replace("_ground-truth-files_dataset.csv", "")
    # Split into name + version
    m = re.match(r"(.+)-([\w\.\_]+)$", base)
    if m:
        return m.group(1), m.group(2)
    else:
        return base, "unknown"

rows = []

for df_old, df_new, name1, name2 in dataset_pairs:
    dataset_name_old, version_old = extract_version(name1)
    dataset_name_new, version_new = extract_version(name2)

    # Compute metrics for old
    files_old = df_old.shape[0]
    defective_old = df_old["Bug"].sum()
    benign_old = files_old - defective_old

    # Compute metrics for new
    files_new = df_new.shape[0]
    defective_new = df_new["Bug"].sum()
    benign_new = files_new - defective_new

    # Append rows
    rows.append({
        "Dataset": dataset_name_old,
        "Version": version_old,
        "#Files": files_old,
        "#Defective": defective_old/files_old * 100,
        "#Benign": benign_old/files_old * 100
    })
    rows.append({
        "Dataset": dataset_name_new,
        "Version": version_new,
        "#Files": files_new,
        "#Defective": defective_new/files_new * 100,
        "#Benign": benign_new/files_new * 100
    })

# Build final dataframe
summary_df = pd.DataFrame(rows)

summary_df

,Dataset,Version,#Files,#Defective,#Benign
0,lucene,2.9.0,1368,8.406433,91.593567
1,lucene,3.0.0,1337,5.609574,94.390426
2,groovy,1_5_7,757,2.113606,97.886394
3,groovy,1_6_BETA_2,884,3.619910,96.380090
4,hbase,0.95.0,1669,10.665069,89.334931
5,hbase,0.95.2,1834,8.887677,91.112323
6,camel,2.10.0,7914,1.756381,98.243619
7,camel,2.11.0,8846,1.526113,98.473887
8,jruby,1.4.0,978,12.985685,87.014315
9,jruby,1.5.0,1131,2.210433,97.789567


In [12]:
import javalang
import pandas as pd
from difflib import SequenceMatcher
from tqdm import tqdm
import random
from rapidfuzz import fuzz

def sanitize_java_source(src):
    """
    Remove leading non-Java lines (e.g., # comments).
    Keeps everything from first 'package' or 'import' or class/interface declaration.
    """
    if not isinstance(src, str):
        return src
    lines = src.splitlines()
    cleaned = []
    for line in lines:
        stripped = line.strip()
        # Allow typical Java start lines
        if stripped.startswith(("#", "import ", "package ", "class ", "interface ", "enum ")):
            cleaned.append(line)
        elif cleaned:
            # Once Java code started, keep everything
            cleaned.append(line)
    return "\n".join(cleaned)

# --- AST extraction with caching for speed ---
AST_CACHE = {}

def get_java_ast_sequence(source_code):
    """
    Parse Java source code into a sequence of AST node types.
    Caches results for speed. Ignores files that fail parsing.
    """
    if not isinstance(source_code, str) or not source_code.strip():
        return None

    key = hash(source_code)
    if key in AST_CACHE:
        return AST_CACHE[key]

    # Preprocess to remove non-Java leading lines
    source_code = sanitize_java_source(source_code)

    try:
        tree = javalang.parse.parse(source_code)
        nodes = [type(node).__name__ for _, node in tree]
        ast_seq = " ".join(nodes)
    except (javalang.parser.JavaSyntaxError, javalang.tokenizer.LexerError, TypeError):
        ast_seq = None

    AST_CACHE[key] = ast_seq
    return ast_seq

# --- Exact path matching ---
def exact_path_matches(df_old, df_new):
    commons = set(df_old.File) & set(df_new.File)
    return pd.DataFrame({
        "df_new_file": list(commons),
        "df_old_file": list(commons),
        "similarity": [1.0] * len(commons)
    })


# --- Remove exact matches before AST diff ---
def remove_exact_matches(df_old, df_new, exact_df):
    old_used = set(exact_df.df_old_file)
    new_used = set(exact_df.df_new_file)
    df_old_rem = df_old[~df_old.File.isin(old_used)].reset_index(drop=True)
    df_new_rem = df_new[~df_new.File.isin(new_used)].reset_index(drop=True)
    return df_old_rem, df_new_rem


# --- Greedy AST similarity matching ---
def match_ast_files(df_old, df_new, similarity_threshold=0.9):
    """
    Greedy one-to-one matching using AST node sequences.
    """

    df_old = df_old.dropna(subset=["SRC"]).reset_index(drop=True)
    df_new = df_new.dropna(subset=["SRC"]).reset_index(drop=True)

    df_old = df_old.copy()
    df_new = df_new.copy()

    # Compute AST sequences once
    df_old["AST"] = df_old["SRC"].apply(get_java_ast_sequence)
    df_new["AST"] = df_new["SRC"].apply(get_java_ast_sequence)

    used_old = set()
    matches = []
    
    max_matches = min(len(df_old), len(df_new))
    
    for _, new_row in tqdm(df_new.iterrows(), total=len(df_new)):
        if len(used_old) >= max_matches:
            break
        if new_row.AST is None:
            continue
    
        best_score = 0.0
        best_idx = None
    
        old_indices = list(df_old.index)
        random.shuffle(old_indices)
        cutoff = max(1, int(1.00 * len(old_indices)))  # 27% of candidates
    
        for idx in old_indices[:cutoff]:
            old_row = df_old.loc[idx]
            if idx in used_old or old_row.AST is None:
                continue
    
            # Fast approximate similarity
            score = fuzz.ratio(new_row.AST, old_row.AST) / 100.0
            if score > best_score:
                best_score = score
                best_idx = idx
    
        if best_score >= similarity_threshold:
            used_old.add(best_idx)
            matches.append({
                "df_new_file": new_row.File,
                "df_old_file": df_old.loc[best_idx, "File"],
                "similarity": best_score
            })

    matches_df = pd.DataFrame(matches)

    if not matches_df.empty:
        assert matches_df.df_new_file.is_unique
        assert matches_df.df_old_file.is_unique

    return matches_df


# --- Full pipeline: exact match + AST diff ---
def match_files_full(df_old, df_new, similarity_threshold=0.8):
    """
    Full file-matching pipeline using:
      1) Exact path match
      2) AST-based structural similarity for remaining files
    """
    exact_df = exact_path_matches(df_old, df_new)

    df_old_rem, df_new_rem = remove_exact_matches(df_old, df_new, exact_df)

    ast_df = match_ast_files(
        df_old_rem,
        df_new_rem,
        similarity_threshold=similarity_threshold
    )

    return pd.concat([exact_df, ast_df], ignore_index=True)

In [13]:
# Dictionary to store matches for each dataset pair
ast_matches_store = ast_matches_store if 'ast_matches_store' in globals() else {}

In [14]:
for name1, name2 in dataset_names:
    # Skip if already computed
    if (name1, name2) in ast_matches_store and not ast_matches_store[(name1, name2)].empty:
        continue

    # Load CSVs
    df_old = pd.read_csv(name1)
    df_new = pd.read_csv(name2)
    
    # Rename columns (just in case)
    df_old = df_old.rename(columns={"File": "File", "SRC": "SRC", "Bug": "Bug"})
    df_new = df_new.rename(columns={"File": "File", "SRC": "SRC", "Bug": "Bug"})
    
    # Compute and store matches
    matches_df = match_files_full(df_old, df_new)
    
    # Key by dataset pair
    ast_matches_store[(name1, name2)] = matches_df

100%|██████████| 555/555 [00:48<00:00, 11.53it/s]


In [19]:
tmp_df = ast_matches_store[dataset_names[5]]
tmp_df[tmp_df['df_new_file']!=tmp_df['df_old_file']]

,df_new_file,df_old_file,similarity
2150,java/build/org/apache/derbyBuild/MessageBundle...,java/testing/org/apache/derbyTesting/functionT...,0.913508
2151,java/engine/org/apache/derby/iapi/error/ErrorS...,java/engine/org/apache/derby/iapi/services/con...,1.000000
2152,java/engine/org/apache/derby/iapi/error/Shutdo...,java/engine/org/apache/derby/iapi/services/con...,1.000000
2153,java/stubs/jdbc4/javax/sql/RowSetEvent.java,java/testing/org/apache/derbyTesting/functionT...,0.803324
2154,java/testing/org/apache/derbyTesting/functionT...,java/testing/org/apache/derbyTesting/functionT...,0.823063


In [16]:
import pandas as pd
from tqdm import tqdm

def fmt(n, pct):
    """Format as: number (xx.xx%)"""
    return f"{n} ({pct*100:.2f}%)"

def build_merged_df(df_old, df_new, matches_df):
    """
    Build a merged dataframe using AST matches.
    """
    # Index for fast lookup
    df_old_i = df_old.set_index("File")
    df_new_i = df_new.set_index("File")

    rows = []
    for _, row in matches_df.iterrows():
        old_f = row["df_old_file"]
        new_f = row["df_new_file"]

        rows.append({
            "old_file": old_f,
            "new_file": new_f,
            "similarity": row["similarity"],

            "SRC_old": df_old_i.loc[old_f, "SRC"],
            "SRC_new": df_new_i.loc[new_f, "SRC"],

            "Bug_old": df_old_i.loc[old_f, "Bug"],
            "Bug_new": df_new_i.loc[new_f, "Bug"],
        })

    return pd.DataFrame(rows)

def analyze_datasets(dataset_names, ast_matches_store, 
                     dataset_names_final=None,
                    # similarity_threshold=0.5
                    ):
    results = []

    for name1, name2 in tqdm(dataset_names):
        dataset = name1.split("-")[0]
        past_ver = name1.replace(f"{dataset}-", "").replace("_ground-truth-files_dataset.csv", "")
        present_ver = name2.replace(f"{dataset}-", "").replace("_ground-truth-files_dataset.csv", "")

        # Load data
        df_old = pd.read_csv(name1)[["File", "SRC", "Bug"]]
        df_new = pd.read_csv(name2)[["File", "SRC", "Bug"]]

        # AST matches (precomputed)
        matches_df = ast_matches_store[(name1, name2)]

        # Build merged dataframe
        merged = build_merged_df(df_old, df_new, matches_df)

        # merged = merged[merged['similarity'] > similarity_threshold]

        # ---- Core counts ----
        commons = len(merged)
        exact_same = (merged["SRC_old"] == merged["SRC_new"]).sum()

        # Changed only
        changed = merged[merged["SRC_old"] != merged["SRC_new"]]

        # Bug transitions
        benign_00 = ((changed.Bug_old == 0) & (changed.Bug_new == 0)).sum()
        benign_10 = ((changed.Bug_old == 1) & (changed.Bug_new == 0)).sum()
        defective_01 = ((changed.Bug_old == 0) & (changed.Bug_new == 1)).sum()
        defective_11 = ((changed.Bug_old == 1) & (changed.Bug_new == 1)).sum()

        # Added / removed (AST-aware)
        added = len(set(df_new.File) - set(merged.new_file))
        removed = len(set(df_old.File) - set(merged.old_file))

        # Denominators
        df_new_len = len(df_new) if len(df_new) else 1
        changed_len = df_new_len

        results.append({
            "dataset": dataset,
            "past_ver": past_ver,
            "present_ver": present_ver,

            "size": f"{len(df_old)}, {len(df_new)}",

            "commons": fmt(commons, commons / df_new_len),
            "added": fmt(added, added / df_new_len),
            "removed": fmt(removed, removed / df_new_len),

            "exact_same": fmt(exact_same, exact_same / df_new_len),

            "benign_00": fmt(benign_00, benign_00 / changed_len),
            "benign_10": fmt(benign_10, benign_10 / changed_len),
            "defective_01": fmt(defective_01, defective_01 / changed_len),
            "defective_11": fmt(defective_11, defective_11 / changed_len),
        })

    df = pd.DataFrame(results)
    if dataset_names_final:
        df = df[df["dataset"].isin(dataset_names_final)]
    return df

In [17]:
analyze_datasets(dataset_names,
                ast_matches_store= ast_matches_store)

100%|██████████| 6/6 [00:03<00:00,  1.82it/s]


,dataset,past_ver,present_ver,size,commons,added,removed,exact_same,benign_00,benign_10,defective_01,defective_11
0,lucene,2.9.0,3.0.0,"1368, 1337",1321 (98.80%),16 (1.20%),47 (3.52%),321 (24.01%),864 (64.62%),65 (4.86%),25 (1.87%),46 (3.44%)
1,groovy,1_5_7,1_6_BETA_2,"757, 884",750 (84.84%),134 (15.16%),7 (0.79%),581 (65.72%),148 (16.74%),3 (0.34%),10 (1.13%),8 (0.90%)
2,hbase,0.95.0,0.95.2,"1669, 1834",1605 (87.51%),229 (12.49%),64 (3.49%),705 (38.44%),657 (35.82%),130 (7.09%),65 (3.54%),48 (2.62%)
3,camel,2.10.0,2.11.0,"7914, 8846",7894 (89.24%),952 (10.76%),20 (0.23%),5986 (67.67%),1724 (19.49%),110 (1.24%),54 (0.61%),20 (0.23%)
4,jruby,1.4.0,1.5.0,"978, 1131",969 (85.68%),162 (14.32%),9 (0.80%),611 (54.02%),230 (20.34%),108 (9.55%),6 (0.53%),14 (1.24%)
5,derby,10.3.1.4,10.5.1.1,"2206, 2705",2155 (79.67%),550 (20.33%),51 (1.89%),1319 (48.76%),531 (19.63%),191 (7.06%),20 (0.74%),94 (3.48%)


In [26]:
def evaluate_naive_with_matches(df_old, df_new, matches_df,
                                old_file_col="File", new_file_col="File",
                                label_col="Bug"):
    """
    Naive version-to-version classifier using matched files.
    - df_old: previous version
    - df_new: current version
    - matches_df: DataFrame with df_new_file, df_old_file, similarity
    """

    # Index old version by real file names for label lookup
    old_labels = dict(zip(df_old["File"], df_old[label_col]))

    y_true = []
    y_pred = []

    for _, row in df_new.iterrows():
        file_new = row[new_file_col]
        true_label = row[label_col]

        # Check if this new file has a match
        match_row = matches_df[matches_df['df_new_file'] == file_new]

        if not match_row.empty:
            # Use the corresponding old file
            file_old = match_row.iloc[0]['df_old_file']
            pred_label = old_labels.get(file_old, 0)  # fallback to benign if somehow missing
        else:
            pred_label = 0  # new file → predict benign

        y_true.append(true_label)
        y_pred.append(pred_label)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # ------------------- Metrics -------------------
    prec = precision_score(y_true, y_pred, zero_division=0, average='macro')
    rec  = recall_score(y_true, y_pred, zero_division=0, average='macro')
    f1   = f1_score(y_true, y_pred, zero_division=0, average='macro')
    mcc  = matthews_corrcoef(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(tpr * tnr)

    try:
        auc = roc_auc_score(y_true, y_pred, average='macro')
    except ValueError:
        auc = np.nan

    try:
        aucpr = average_precision_score(y_true, y_pred)
    except ValueError:
        aucpr = np.nan

    return {
        "Prec": prec,
        "Rec": rec,
        "FPR": fpr,
        "AUC": auc,
        "AUCPR": aucpr,
        "F1": f1,
        "MCC": mcc,
        "G-Mean": gmean
    }

In [27]:
results = []

for df_old, df_new, name1, name2 in dataset_pairs:
    matches_df = ast_matches_store.get((name1, name2), pd.DataFrame(columns=['df_new_file', 'df_old_file', 'similarity']))

    metrics = evaluate_naive_with_matches(
        df_old, df_new, matches_df,
        new_file_col="File"  # column in df_new
    )

    results.append({
        "Dataset": name2.replace("_ground-truth-files_dataset.csv", ""),
        **metrics
    })

results_df = pd.DataFrame(results)

In [28]:
results_df

,Dataset,Prec,Rec,FPR,AUC,AUCPR,F1,MCC,G-Mean
0,lucene-3.0.0,0.702405,0.800518,0.052298,0.800518,0.297823,0.739376,0.493260,0.786871
1,groovy-1_6_BETA_2,0.831653,0.668941,0.005869,0.668941,0.260084,0.721609,0.473412,0.584579
2,hbase-0.95.2,0.600109,0.608340,0.077798,0.608340,0.142114,0.603943,0.208287,0.521123
3,camel-2.11.0,0.565339,0.567244,0.013661,0.567244,0.034317,0.566276,0.132569,0.382262
4,jruby-1.5.0,0.549640,0.728915,0.102170,0.728915,0.071458,0.562721,0.213198,0.709073
5,derby-10.5.1.1,0.652974,0.799392,0.092126,0.799392,0.245186,0.693123,0.428015,0.791996


In [14]:
import pandas as pd
from difflib import SequenceMatcher
from tqdm import tqdm

# -------------------------------------------------------
# 1. Similarity functions
# -------------------------------------------------------

def line_overlap_sim(c1: str, c2: str):
    """Similarity = 2 * |intersection(lines)| / (|lines1| + |lines2|)."""
    s1 = set(c1.splitlines())
    s2 = set(c2.splitlines())
    if not s1 and not s2:
        return 1.0
    inter = len(s1 & s2)
    return (2 * inter) / (len(s1) + len(s2))

def fuzzy_sim(c1: str, c2: str):
    """Fuzzy similarity via difflib."""
    return SequenceMatcher(None, c1, c2).ratio()

# choose similarity function
SIM = line_overlap_sim
# SIM = line_overlap_sim   # alternative


# -------------------------------------------------------
# 2. Filename-based matching
# -------------------------------------------------------

def filename_match(df1, df2):
    """
    Given df1 (V1) and df2 (V2), return:
        matched_df2   df of V2 rows with direct name match
        unmatched_df2 df of V2 rows with no name match
        unmatched_df1 df of V1 rows not matched by name
    """

    # Map filename → full row in V1
    name_to_row_V1 = {row.File: row for _, row in df1.iterrows()}

    matched_rows = []
    unmatched_V2_rows = []
    used_V1_indices = set()

    for idx2, r2 in df2.iterrows():
        if r2.File in name_to_row_V1:
            r1 = name_to_row_V1[r2.File]
            matched_rows.append({
                "V2_idx": idx2,
                "V1_idx": r1.name,
                "V2_file": r2.File,
                "V1_file": r1.File
            })
            used_V1_indices.add(r1.name)
        else:
            unmatched_V2_rows.append(idx2)

    unmatched_V1_rows = [i for i in df1.index if i not in used_V1_indices]

    matched_df2 = pd.DataFrame(matched_rows)
    unmatched_df2 = df2.loc[unmatched_V2_rows]
    unmatched_df1 = df1.loc[unmatched_V1_rows]

    return matched_df2, unmatched_df2, unmatched_df1


# -------------------------------------------------------
# 3. Similarity-based matching for unmatched files
# -------------------------------------------------------

import re

def strip_comments(code: str):
    code = re.sub(r"//.*?$", "", code, flags=re.MULTILINE)
    code = re.sub(r"#.*?$", "", code, flags=re.MULTILINE)
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    return code.strip()
import numpy as np
from tqdm import tqdm

def similarity_match(unmatched_df2, unmatched_df1):
    results = []

    V2_rows = list(unmatched_df2.itertuples())
    V1_rows = list(unmatched_df1.itertuples())

    for y in tqdm(V2_rows):
        y_clean = strip_comments(y.SRC)
        sims = []
        for x in V1_rows:
            x_clean = strip_comments(x.SRC)
            s = SIM(x_clean, y_clean)
            sims.append((x, s, x_clean))

        sims_sorted = sorted(sims, key=lambda p: p[1], reverse=True)

        (x1, s1, x1_clean) = sims_sorted[0]
        if len(sims_sorted) > 1:
            (x2, s2, _) = sims_sorted[1]
        else:
            x2, s2 = None, 0

        gaps = [abs(sims_sorted[i][1] - sims_sorted[i+1][1])
                for i in range(len(sims_sorted)-1)]

        avg_gap = np.mean(gaps) if gaps else 0
        gap_sigma = np.std(gaps) if gaps else 0

        results.append({
            "y_idx": y.Index,
            "y_file": y.File,
            "y_src": y.SRC,
            "y_src_clean": y_clean,

            "top1_file": x1.File,
            "top1_src": x1.SRC,
            "top1_src_clean": x1_clean,
            "top1_score": s1,

            "top2_file": None if x2 is None else x2.File,
            "top2_score": s2,

            "top1_gap": s1 - s2,
            "avg_gap": avg_gap,
            "gap_sigma": gap_sigma
        })

    return pd.DataFrame(results)

# -------------------------------------------------------
# 4. Main Wrapper
# -------------------------------------------------------

def run_df_matching(df1, df2):
    print("Running filename matching…")
    matched_df2, U2, U1 = filename_match(df1, df2)

    print(f"Filename matches: {len(matched_df2)}")
    print(f"Unmatched V2: {len(U2)}   Unmatched V1: {len(U1)}")

    print("Running similarity matching…")
    df_sim = similarity_match(U2, U1)

    return matched_df2, df_sim

In [15]:
def apply_final_match_rules(sim_df, sim_threshold=0.8, gap_times=1.0):
    df = sim_df.copy()

    def decide(row):
        cond1 = row["top1_score"] > sim_threshold

        cond2 = row["top1_gap"] >= row["avg_gap"] + gap_times * row["gap_sigma"]

        if cond1 and cond2:
            return row["top1_file"]
        return None

    df["final_match"] = df.apply(decide, axis=1)
    num_matches = df["final_match"].notna().sum()

    print(f"Accepted similarity-based matches: {num_matches}")
    return df

In [ ]:
# ============================================================
# Clean all datasets by dropping rows with NaN in ANY column
# ============================================================
cleaned_dataset_pairs = []

for (df_old, df_new, name1, name2) in dataset_pairs:
    df_old_clean = df_old.dropna().reset_index(drop=True)
    df_new_clean = df_new.dropna().reset_index(drop=True)
    cleaned_dataset_pairs.append((df_old_clean, df_new_clean, name1, name2))

# Replace dataset_pairs with the cleaned version
dataset_pairs = cleaned_dataset_pairs